In [1]:
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'squad'))

import pandas as pd, numpy as np
from scipy.stats import spearmanr

import assembly
importlib.reload(assembly)

print("Assembly module loaded.")
print("Available:", [f for f in dir(assembly) if not f.startswith('_')][:15])

Assembly module loaded.
Available: ['BASE', 'CS_PTS', 'DC_BASE', 'GOAL_PTS', 'LEAGUE_AVG_LAMBDA', 'TEAM_MAP', 'build_predictions', 'get_bonus_model', 'get_dc_2526', 'get_fixtures_2526', 'get_minutes_2526', 'get_rates_2526', 'np', 'pd', 'spearmanr']


In [2]:
import time

t0 = time.time()
mins_out = assembly.get_minutes_2526()
print(f"minutes:   {time.time()-t0:.1f}s")

t0 = time.time()
rates, priors = assembly.get_rates_2526("2025")
print(f"rates:     {time.time()-t0:.1f}s")

t0 = time.time()
fixtures = assembly.get_fixtures_2526()
print(f"dixon-coles: {time.time()-t0:.1f}s")

t0 = time.time()
dc_out = assembly.get_dc_2526()
print(f"defensive: {time.time()-t0:.1f}s")

t0 = time.time()
bps_model, bps_to_bonus, BPS_FEATURES = assembly.get_bonus_model()
print(f"bonus:     {time.time()-t0:.1f}s")

minutes:   17.8s
rates:     0.0s
dixon-coles: 5.7s
defensive: 4.9s
bonus:     0.7s


In [3]:
# Assemble-only version: takes pre-computed component outputs, returns predictions.
# This is assembly.build_predictions() with the component calls lifted OUT,
# so the walk-forward loop can supply per-gameweek components.

BASE = assembly.BASE
GOAL_PTS, CS_PTS = assembly.GOAL_PTS, assembly.CS_PTS
LEAGUE_AVG_LAMBDA, TEAM_MAP, DC_BASE = assembly.LEAGUE_AVG_LAMBDA, assembly.TEAM_MAP, assembly.DC_BASE

def assemble(df, cw, mins_out, rates, priors, fixtures, dc_out,
             bps_model, bps_to_bonus, BPS_FEATURES, bonus_mean, gws=None):
    """Assemble E[points] from pre-computed component outputs.
    gws: optional list of gameweeks to restrict to (for walk-forward).
    bonus_mean: the actual-bonus mean used for recalibration (caller supplies it,
                so walk-forward can pass a PRIOR-ONLY mean instead of full-season)."""
    v = df[(df["season"] == "2025-26") & (df["position"] != "AM")].copy()
    skel = v[["element", "GW", "name", "position", "team", "minutes", "total_points"]].copy()
    skel["element"] = pd.to_numeric(skel["element"], errors="coerce").astype(int)
    skel = skel.rename(columns={"GW": "gw", "total_points": "actual_points"})
    if gws is not None:
        skel = skel[skel["gw"].isin(gws)]
    skel = skel.merge(cw[["element", "player_id", "understat_id"]], on="element", how="left")
    asm = (skel.sort_values(["element", "gw"]).groupby(["element", "gw"], as_index=False)
           .agg({"name": "first", "position": "first", "team": "first", "minutes": "sum",
                 "actual_points": "sum", "player_id": "first", "understat_id": "first"}))

    asm = asm.merge(mins_out[["element", "gw", "p_start", "p60", "e_minutes"]],
                    on=["element", "gw"], how="left")

    rates_clean = rates.sort_values("npxg90", ascending=False).drop_duplicates("understat_id", keep="first")
    asm["understat_id_num"] = pd.to_numeric(asm["understat_id"], errors="coerce")
    rates_clean = rates_clean.copy()
    rates_clean["understat_id"] = pd.to_numeric(rates_clean["understat_id"], errors="coerce")
    asm = asm.merge(rates_clean, left_on="understat_id_num", right_on="understat_id",
                    how="left", suffixes=("", "_r"))
    pos_lab = {"FWD": "F", "MID": "M", "DEF": "D", "GK": "D"}
    def fb(row, stat):
        return priors.get(pos_lab.get(row["position"], "M"), priors["M"])[stat]
    nn = asm["npxg90"].isna(); nx = asm["xa90"].isna()
    if nn.any(): asm.loc[nn, "npxg90"] = asm[nn].apply(lambda r: fb(r, "npxg"), axis=1)
    if nx.any(): asm.loc[nx, "xa90"] = asm[nx].apply(lambda r: fb(r, "xa"), axis=1)
    assert asm.duplicated(["element", "gw"]).sum() == 0

    fx = fixtures.copy()
    fx["home"] = fx["home"].map(lambda t: TEAM_MAP.get(t, t))
    fx["away"] = fx["away"].map(lambda t: TEAM_MAP.get(t, t))
    home = fx[["home", "match_date", "lam_home", "lam_away", "p_home_cs"]].copy()
    home.columns = ["team", "match_date", "team_lambda", "opp_lambda", "p_cs"]
    away = fx[["away", "match_date", "lam_away", "lam_home", "p_away_cs"]].copy()
    away.columns = ["team", "match_date", "team_lambda", "opp_lambda", "p_cs"]
    tf = pd.concat([home, away], ignore_index=True)
    vv = df[df["season"] == "2025-26"].copy()
    vv["match_date"] = pd.to_datetime(vv["kickoff_time"]).dt.date
    tgd = vv[["team", "GW", "match_date"]].drop_duplicates().rename(columns={"GW": "gw"})
    tf["match_date"] = pd.to_datetime(tf["match_date"]).dt.date
    tgd["match_date"] = pd.to_datetime(tgd["match_date"]).dt.date
    tf = tf.merge(tgd, on=["team", "match_date"], how="left")
    asm = asm.merge(tf[["team", "gw", "team_lambda", "opp_lambda", "p_cs"]].drop_duplicates(["team", "gw"]),
                    on=["team", "gw"], how="left")

    asm = asm.merge(dc_out[["player_id", "gw", "p_dc_hit"]], on=["player_id", "gw"], how="left")
    need = asm["p_dc_hit"].isna()
    asm.loc[need, "p_dc_hit"] = asm.loc[need, "position"].map(DC_BASE).fillna(0.10)
    asm = (asm.sort_values("p_dc_hit", ascending=False).drop_duplicates(["element", "gw"], keep="first")
           .sort_values(["element", "gw"]).reset_index(drop=True))

    a = asm.copy()
    a["minutes_frac"] = (a["e_minutes"] / 90.0).clip(0, 1)
    a["fixture_scale"] = (a["team_lambda"] / LEAGUE_AVG_LAMBDA).fillna(1.0).clip(0.5, 2.0)
    a["e_goals"] = a["npxg90"] * a["minutes_frac"] * a["fixture_scale"]
    a["e_assists"] = a["xa90"] * a["minutes_frac"] * a["fixture_scale"]
    a["pts_goals"] = a["e_goals"] * a["position"].map(GOAL_PTS)
    a["pts_assists"] = a["e_assists"] * 3
    a["p_60plus"] = a["p_start"] * a["p60"]
    a["p_play_any"] = a["p_start"] + (1 - a["p_start"]) * 0.30
    a["pts_appear"] = a["p_60plus"] * 2 + (a["p_play_any"] - a["p_60plus"]).clip(lower=0) * 1
    a["pts_cs"] = a["p_cs"] * a["position"].map(CS_PTS) * a["p_60plus"]
    a["pts_dc"] = a["p_dc_hit"] * 2 * a["minutes_frac"]
    a["e_points_core"] = a["pts_appear"] + a["pts_goals"] + a["pts_assists"] + a["pts_cs"] + a["pts_dc"]

    bps_input = pd.DataFrame({
        "goals_scored": a["e_goals"], "assists": a["e_assists"],
        "clean_sheets": a["p_cs"] * a["p_60plus"], "minutes": a["e_minutes"],
        "is_def": (a["position"] == "DEF").astype(int), "is_mid": (a["position"] == "MID").astype(int),
        "is_gk": (a["position"] == "GK").astype(int),
        "saves": 0, "yellow_cards": 0, "red_cards": 0, "goals_conceded": 0,
        "penalties_missed": 0, "own_goals": 0})
    a["pred_bps"] = bps_model.predict(bps_input[BPS_FEATURES])
    a["exp_bonus"] = bps_to_bonus(a["pred_bps"].values) * a["minutes_frac"]
    a["exp_bonus"] *= bonus_mean / a["exp_bonus"].mean()   # caller supplies the mean
    a["e_points"] = a["e_points_core"] + a["exp_bonus"]
    return a

print("assemble() defined")

assemble() defined


In [4]:
# Load the shared inputs
df = pd.read_parquet(BASE + r"\data\history\all_seasons_fixed.parquet")
cw = pd.read_csv(BASE + r"\data\history\player_id_crosswalk_final.csv")

# Full-season bonus mean (same as assembly.py uses) — for this verification only
bonus_mean = pd.to_numeric(df[df["season"] == "2025-26"]["bonus"], errors="coerce").mean()

a = assemble(df, cw, mins_out, rates, priors, fixtures, dc_out,
             bps_model, bps_to_bonus, BPS_FEATURES, bonus_mean)

val = a.dropna(subset=["actual_points", "e_points"]).copy()
val["actual_points"] = pd.to_numeric(val["actual_points"], errors="coerce")
print(f"Rows: {len(val)}")
print(f"Spearman: {spearmanr(val['e_points'], val['actual_points']).correlation:.3f}")
print(f"MAE:      {(val['e_points'] - val['actual_points']).abs().mean():.2f}")
print(f"Mean pred/actual: {val['e_points'].mean():.2f} / {val['actual_points'].mean():.2f}")

Rows: 29338
Spearman: 0.715
MAE:      1.14
Mean pred/actual: 1.38 / 1.17


In [5]:
import time

def walk_forward(df, cw, gw_list=None, verbose=True):
    """Walk-forward harness (SKELETON: components not yet GW-aware).
    Assembles one gameweek at a time and stitches the results."""
    if gw_list is None:
        gw_list = sorted(df[df["season"] == "2025-26"]["GW"].dropna().unique().astype(int))

    out = []
    for k in gw_list:
        # --- components (STEP 1: still full-data; cutoffs come in later steps) ---
        m_k, r_k, p_k, f_k, d_k = mins_out, rates, priors, fixtures, dc_out
        bm_k = bonus_mean   # TODO: prior-only mean when bonus becomes GW-aware

        a_k = assemble(df, cw, m_k, r_k, p_k, f_k, d_k,
                       bps_model, bps_to_bonus, BPS_FEATURES, bm_k, gws=[k])
        out.append(a_k)
        if verbose and k % 10 == 0:
            print(f"  GW{k}: {len(a_k)} rows")

    return pd.concat(out, ignore_index=True)


t0 = time.time()
wf = walk_forward(df, cw)
print(f"\nDone in {time.time()-t0:.1f}s — {len(wf)} rows")

v = wf.dropna(subset=["actual_points", "e_points"]).copy()
v["actual_points"] = pd.to_numeric(v["actual_points"], errors="coerce")
print(f"Spearman: {spearmanr(v['e_points'], v['actual_points']).correlation:.3f}")
print(f"MAE:      {(v['e_points'] - v['actual_points']).abs().mean():.2f}")

  GW10: 747 rows
  GW20: 790 rows
  GW30: 822 rows

Done in 3.7s — 29338 rows
Spearman: 0.715
MAE:      1.14


In [7]:
import sys, importlib, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'squad'))

import pandas as pd, numpy as np
from scipy.stats import spearmanr

import minutes as minutes_mod
importlib.reload(minutes_mod)

# Baseline: prior seasons only, GW10
m_base = minutes_mod.get_minutes(up_to_gw=None, predict_gws=[10])
# Walk-forward: prior seasons + 2025-26 GW1-9, predicting GW10
m_wf = minutes_mod.get_minutes(up_to_gw=10, predict_gws=[10])

print(f"Baseline rows: {len(m_base)}   WF rows: {len(m_wf)}")
print(f"Baseline mean e_minutes: {m_base['e_minutes'].mean():.3f}")
print(f"WF       mean e_minutes: {m_wf['e_minutes'].mean():.3f}")

cmp = m_base[["element","e_minutes"]].merge(
    m_wf[["element","e_minutes"]], on="element", suffixes=("_base","_wf"))
diff = (cmp["e_minutes_wf"] - cmp["e_minutes_base"]).abs()
print(f"\nMean abs difference: {diff.mean():.3f}")
print(f"Max  abs difference: {diff.max():.3f}")
print(f"Identical: {(diff < 1e-9).sum()} / {len(cmp)}")

Baseline rows: 747   WF rows: 747
Baseline mean e_minutes: 26.797
WF       mean e_minutes: 26.695

Mean abs difference: 0.893
Max  abs difference: 12.754
Identical: 0 / 747


In [8]:
import sys, importlib, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'squad'))
import pandas as pd, numpy as np
from scipy.stats import spearmanr

import minutes as minutes_mod, bonus as bonus_mod, dixon_coles as dc_mod
import attacking_rates as rates_mod, defensive as def_mod
for m in [minutes_mod, bonus_mod, dc_mod, rates_mod, def_mod]:
    importlib.reload(m)

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"
df = pd.read_parquet(BASE + r"\data\history\all_seasons_fixed.parquet")
cw = pd.read_csv(BASE + r"\data\history\player_id_crosswalk_final.csv")

# GW -> earliest kickoff date (the deadline proxy / DC cutoff)
v25 = df[df["season"] == "2025-26"].copy()
v25["kick"] = pd.to_datetime(v25["kickoff_time"])
gw_start = v25.groupby("GW")["kick"].min().sort_index()
print("GW date map built:", len(gw_start), "gameweeks")
print(gw_start.head(3))

GW date map built: 38 gameweeks
GW
1   2025-08-15 19:00:00+00:00
2   2025-08-22 19:00:00+00:00
3   2025-08-30 11:30:00+00:00
Name: kick, dtype: datetime64[us, UTC]


In [10]:
def walk_forward_strict(gw_list, verbose=True):
    """Strict walk-forward: every component retrained/refit using only data
    strictly before each gameweek."""
    rates, priors = rates_mod.get_rates("2025-26")
    dc_out = def_mod.get_dc_2526()

    out = []
    for k in gw_list:
        t0 = time.time()
        cutoff = gw_start.loc[k].tz_localize(None)   # <-- strip tz to match odds file

        m_k = minutes_mod.get_minutes(up_to_gw=k, predict_gws=[k])
        bps_model, bps_to_bonus, BPS_FEATURES, bonus_mean = bonus_mod.get_bonus_model(up_to_gw=k)
        f_k = dc_mod.get_fixtures(cutoff_date=cutoff)

        a_k = assemble(df, cw, m_k, rates, priors, f_k, dc_out,
                       bps_model, bps_to_bonus, BPS_FEATURES, bonus_mean, gws=[k])
        out.append(a_k)
        if verbose:
            print(f"  GW{k}: {len(a_k)} rows, {time.time()-t0:.1f}s")

    return pd.concat(out, ignore_index=True)


t0 = time.time()
wf_test = walk_forward_strict([10, 20, 30])
print(f"\nTest done in {time.time()-t0:.1f}s — {len(wf_test)} rows")

  GW10: 747 rows, 21.2s
  GW20: 790 rows, 21.0s
  GW30: 822 rows, 21.4s

Test done in 69.3s — 2359 rows


In [11]:
v = wf_test.dropna(subset=["actual_points", "e_points"]).copy()
v["actual_points"] = pd.to_numeric(v["actual_points"], errors="coerce")

print(f"Rows: {len(v)}")
print(f"Spearman: {spearmanr(v['e_points'], v['actual_points']).correlation:.3f}")
print(f"MAE:      {(v['e_points'] - v['actual_points']).abs().mean():.2f}")
print(f"Mean pred/actual: {v['e_points'].mean():.2f} / {v['actual_points'].mean():.2f}")

print("\nPer-gameweek:")
for k in [10, 20, 30]:
    s = v[v["gw"] == k]
    print(f"  GW{k}: Spearman {spearmanr(s['e_points'], s['actual_points']).correlation:.3f}, "
          f"MAE {(s['e_points']-s['actual_points']).abs().mean():.2f}")

Rows: 2359
Spearman: 0.732
MAE:      1.11
Mean pred/actual: 1.39 / 1.16

Per-gameweek:
  GW10: Spearman 0.757, MAE 1.10
  GW20: Spearman 0.728, MAE 1.12
  GW30: Spearman 0.714, MAE 1.10


In [12]:
t0 = time.time()
wf_full = walk_forward_strict(sorted(gw_start.index.astype(int)), verbose=True)
print(f"\nFULL RUN done in {(time.time()-t0)/60:.1f} min — {len(wf_full)} rows")

# save so we never have to re-run it
wf_full.to_parquet(BASE + r"\data\walkforward_2526.parquet", index=False)
print("Saved -> data\\walkforward_2526.parquet")

  GW1: 690 rows, 21.1s
  GW2: 705 rows, 20.7s
  GW3: 712 rows, 20.3s
  GW4: 740 rows, 19.0s
  GW5: 741 rows, 20.2s
  GW6: 742 rows, 19.9s
  GW7: 743 rows, 21.4s
  GW8: 745 rows, 21.0s
  GW9: 746 rows, 19.4s
  GW10: 747 rows, 19.5s
  GW11: 752 rows, 19.9s
  GW12: 755 rows, 20.1s
  GW13: 755 rows, 19.9s
  GW14: 758 rows, 19.2s
  GW15: 759 rows, 19.5s
  GW16: 760 rows, 20.1s
  GW17: 770 rows, 19.5s
  GW18: 775 rows, 19.6s
  GW19: 780 rows, 19.5s
  GW20: 790 rows, 21.1s
  GW21: 795 rows, 21.1s
  GW22: 799 rows, 21.3s
  GW23: 803 rows, 21.0s
  GW24: 811 rows, 20.2s
  GW25: 817 rows, 19.2s
  GW26: 817 rows, 20.2s
  GW27: 818 rows, 19.5s
  GW28: 819 rows, 20.0s
  GW29: 820 rows, 20.4s
  GW30: 822 rows, 21.4s
  GW31: 664 rows, 21.5s
  GW32: 826 rows, 21.8s
  GW33: 829 rows, 21.4s
  GW34: 582 rows, 21.0s
  GW35: 832 rows, 20.7s
  GW36: 838 rows, 20.5s
  GW37: 840 rows, 20.7s
  GW38: 841 rows, 20.9s

FULL RUN done in 13.0 min — 29338 rows
Saved -> data\walkforward_2526.parquet


In [13]:
v = wf_full.dropna(subset=["actual_points", "e_points"]).copy()
v["actual_points"] = pd.to_numeric(v["actual_points"], errors="coerce")

print("=== WALK-FORWARD (strict) vs STATIC ===")
print(f"Rows: {len(v)}")
print(f"Spearman: {spearmanr(v['e_points'], v['actual_points']).correlation:.3f}   (static: 0.715)")
print(f"MAE:      {(v['e_points'] - v['actual_points']).abs().mean():.2f}   (static: 1.15)")
print(f"Mean pred/actual: {v['e_points'].mean():.2f} / {v['actual_points'].mean():.2f}   (static: 1.40 / 1.17)")

print("\n=== Three-band slice (house standard) ===")
bands = {
    "All rows":        v["actual_points"].notna(),
    "Played (>0 min)": v["minutes"] > 0,
    "Started (60+)":   v["minutes"] >= 60,
}
print(f"{'Slice':<18}{'N':>7}{'Spearman':>10}{'MAE':>8}")
print("-" * 43)
for label, m in bands.items():
    s = v[m]
    print(f"{label:<18}{len(s):>7}{spearmanr(s['e_points'], s['actual_points']).correlation:>10.3f}"
          f"{(s['e_points']-s['actual_points']).abs().mean():>8.2f}")

=== WALK-FORWARD (strict) vs STATIC ===
Rows: 29338
Spearman: 0.715   (static: 0.715)
MAE:      1.15   (static: 1.15)
Mean pred/actual: 1.40 / 1.17   (static: 1.40 / 1.17)

=== Three-band slice (house standard) ===
Slice                   N  Spearman     MAE
-------------------------------------------
All rows            29338     0.715    1.15
Played (>0 min)     11361     0.337    2.04
Started (60+)        7738     0.099    2.41
